In [ ]:
import pandas as pd
import re
from google.colab import drive

In [ ]:
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/MonografiaI/Artigo/dataset.csv'
df = pd.read_csv(file_path)

# Exibir as primeiras 5 linhas do DataFrame para verificar
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Unnamed: 0,sourceID,targetID,sourceTime,targetTime,sourceText,targetText,sourceAbs,targetAbs,sourceLineText,...,sourceLineAbs,targetLineAbs,lineNums_Text,lineNums_Abs,diffAbs_ins,diffAbs_del,errSet_diffs,ErrSet,errorClang,errorLLVM
0,0,5780006,5780028,2015-08-24 15:41:24,2015-08-24 15:41:35,#include<stdio.h>\n\nint check_prime(int num)\...,#include<stdio.h>\n\nint check_prime(int num)\...,# include < stdio . h >\n\nint TypeKind.FUNCTI...,# include < stdio . h >\n\nint TypeKind.FUNCTI...,if ( num % i != = 0 ) \n,...,if ( TypeKind.INT % TypeKind.INT != = TokenKin...,if ( TypeKind.INT % TypeKind.INT == TokenKind....,10,10,+ ==,- !=\n- =,4;\n+ ==\n- !=\n- =,4;,e158_277497.c:10:18: error: expected expressio...,10-18 Parse Issue: expected expression\n22-18 ...
1,1,6512934,6512979,2015-11-02 15:30:41,2015-11-02 15:31:16,"#include <stdio.h>\nstruct point{\n int x,y...","#include <stdio.h>\nstruct point{\n int x,y...",# include < stdio . h >\nstruct TypeKind.RECOR...,# include < stdio . h >\nstruct TypeKind.RECOR...,\n,...,\n,} \n,39,39,+ },NaN,1;\n+ },1;,e177_307050.c:41:2: error: expected '}'\n}\n ^...,41-2 Parse Issue: expected '}'
2,2,5626836,5626839,2015-08-07 16:04:56,2015-08-07 16:05:05,#include<stdio.h>\nint main(){\n int a=10;\...,#include<stdio.h>\nint main(){\n int a=10;\...,# include < stdio . h >\nint main ( ) {\nint T...,# include < stdio . h >\nint main ( ) {\nint T...,d = ( b * b ) - 4 * a * c \n,...,TypeKind.INT = ( TypeKind.INT * TypeKind.INT )...,TypeKind.INT = ( TypeKind.INT * TypeKind.INT )...,7,7,+ ;,NaN,2;\n+ ;,2;,e_269128.c:7:18: error: expected ';' after exp...,7-18 Parse Issue: expected ';' after expression
3,3,5582231,5582232,2015-08-03 15:43:57,2015-08-03 15:44:08,"#include<stdio.h>\nint main(){\n printf(""Le...","#include<stdio.h>\nint main(){\n printf(""Le...",# include < stdio . h >\nint main ( ) {\nprint...,# include < stdio . h >\nint main ( ) {\nprint...,"printf ( ""Let\\s \""C"" ! ""); \n",...,"printf ( "" \\ \"" "" ! ""); \n","printf ( "" \\ \"" \"" "" ) ; \n",3,3,"+ )\n+ \""\n+ ;","- "");\n- !","33; 18;\n+ )\n+ \""\n+ ;\n- "");\n- !",33; 18;,e148_262425.c: In function 'main':\ne148_26242...,3-24 Parse Issue: expected ')'\n3-25 Lexical o...
4,4,6256038,6256052,2015-09-30 16:41:46,2015-09-30 16:41:55,"#include <stdio.h>\nint main() {\n\tint n,m,i,...","#include <stdio.h>\nint main() {\n\tint n,m,i,...",# include < stdio . h >\nint main ( ) {\nint T...,# include < stdio . h >\nint main ( ) {\nint T...,print f \n,...,TypeKind.INVALID TypeKind.INVALID \n,\n,60,60,NaN,- TypeKind.INVALID,2; 3; 12;\n- TypeKind.INVALID,2; 3; 12;,e169_297971.c:60:2: error: use of undeclared i...,60-2 Semantic Issue: use of undeclared identif...


In [ ]:
#Quantidade de itens -mensagens de erro
df.shape

(21995, 21)

In [ ]:
df.columns

Index(['Unnamed: 0', 'sourceID', 'targetID', 'sourceTime', 'targetTime',
       'sourceText', 'targetText', 'sourceAbs', 'targetAbs', 'sourceLineText',
       'targetLineText', 'sourceLineAbs', 'targetLineAbs', 'lineNums_Text',
       'lineNums_Abs', 'diffAbs_ins', 'diffAbs_del', 'errSet_diffs', 'ErrSet',
       'errorClang', 'errorLLVM'],
      dtype='object')

In [ ]:

def limpeza_leve(msg):
    if pd.isna(msg):
        return ""

    msg = str(msg).lower()

    # remove arquivo:linha:coluna (ex: main.c:10:5:)
    msg = re.sub(r'.*?:\d+:\d+:\s*', '', msg)

    # remove mensagens finais do compilador
    msg = re.sub(
        r'\b\d+\s+warning(s)?\s+and\s+\d+\s+error(s)?\s+generated\.?',
        '',
        msg
    )
    msg = re.sub(
        r'\b\d+\s+error(s)?\s+generated\.?',
        '',
        msg
    )
    msg = re.sub(
        r'\b\d+\s+warning(s)?\s+generated\.?',
        '',
        msg
    )

    # remove setinhas ^
    msg = re.sub(r'\^+', '', msg)

    # normaliza espaços
    msg = re.sub(r'\s+', ' ', msg).strip()

    return msg


In [ ]:
def extrair_primeiro_erro(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto)

    # separa blocos de erro / warning / note
    blocos = re.split(r'\n(?=.*\b(error|warning|note):)', texto)

    for bloco in blocos:
        if "error:" in bloco.lower():
            return limpeza_leve(bloco)

    return ""


In [ ]:
df["mensagem_erro"] = df["errorClang"].apply(extrair_primeiro_erro)
df[["errorClang", "mensagem_erro"]].head()


,errorClang,mensagem_erro
0,e158_277497.c:10:18: error: expected expressio...,error: expected expression if(num%i!==0)
1,e177_307050.c:41:2: error: expected '}'\n}\n ^...,error: expected '}' }
2,e_269128.c:7:18: error: expected ';' after exp...,error: expected ';' after expression d=(b*b)-4...
3,e148_262425.c: In function 'main':\ne148_26242...,"error: expected ')' before '!' token printf(""l..."
4,e169_297971.c:60:2: error: use of undeclared i...,error: use of undeclared identifier 'print'; d...


In [ ]:
df_final = df[[
    "sourceAbs",
    "sourceText",
    "mensagem_erro"
]].copy()

# Coluna de rótulo (para anotação manual)
df_final["rotulo"] = ""

# Verificação
df_final.head()
df_final.shape

(21995, 4)

In [ ]:
df_rotulacao = df_final.sample(n=800, random_state=42)

df_rotulacao.shape
df_rotulacao.head()

,sourceAbs,sourceText,mensagem_erro,rotulo
11480,# include < stdio . h >\n\nint main ( )\n{\nin...,#include<stdio.h>\n\nint main()\n{\n int...,error: expected ';' after expression x=a ;,
5888,# include < stdio . h >\n\nint TypeKind.INT ;\...,#include <stdio.h>\n\nint n;\n\nvoid revstr(ch...,"error: expected ';' after expression printf(""%...",
5186,# include < stdio . h >\nint TypeKind.FUNCTION...,#include <stdio.h>\nint read_line(char str[]){...,error: expected ';' in 'for' statement specifi...,
15357,# include < stdio . h >\n# include < stdlib . ...,#include <stdio.h>\n#include <stdlib.h>\nint n...,clang: error: linker command failed with exit ...,
12560,# include < stdio . h >\nvoid TypeKind.FUNCTIO...,#include <stdio.h>\nvoid check_contigous(int a...,error: redefinition of 'b' with a different ty...,


In [ ]:
df_rotulacao.insert(0, "id", range(1, len(df_rotulacao) + 1))

df_rotulacao.head()

,id,sourceAbs,sourceText,mensagem_erro,rotulo
11480,1,# include < stdio . h >\n\nint main ( )\n{\nin...,#include<stdio.h>\n\nint main()\n{\n int...,error: expected ';' after expression x=a ;,
5888,2,# include < stdio . h >\n\nint TypeKind.INT ;\...,#include <stdio.h>\n\nint n;\n\nvoid revstr(ch...,"error: expected ';' after expression printf(""%...",
5186,3,# include < stdio . h >\nint TypeKind.FUNCTION...,#include <stdio.h>\nint read_line(char str[]){...,error: expected ';' in 'for' statement specifi...,
15357,4,# include < stdio . h >\n# include < stdlib . ...,#include <stdio.h>\n#include <stdlib.h>\nint n...,clang: error: linker command failed with exit ...,
12560,5,# include < stdio . h >\nvoid TypeKind.FUNCTIO...,#include <stdio.h>\nvoid check_contigous(int a...,error: redefinition of 'b' with a different ty...,


In [ ]:
classes = {
    "C1": "Tipos de dados, variáveis e constantes",
    "C2": "Expressões e operadores aritméticos, relacionais e lógicos",
    "C3": "Estruturas condicionais e de repetição",
    "C4": "Modularização de programas (funções e procedimentos)",
    "C5": "Estruturas de dados homogêneas e heterogêneas",
    "C6": "Manipulação de cadeias de caracteres",
    "C7": "Ponteiros e alocação dinâmica de memória"
}


In [ ]:
df_rotulacao.to_csv(
    #"/content/drive/MyDrive/MonografiaI/Artigo/dataset_mensagens_e_codigos.csv",
    index=False
)
